# Relationship Discovery and Approval
Discover registry/custom/trusted relationships, inspect safeguards and confidence, and route ambiguous candidates through the shared approval workflow.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import TablePair, load_app_config, load_business_context, load_table_pairs
from dq_agent.connectors import BigQueryConnector
from dq_agent.context_store import read_context
from dq_agent.context_utils import (
    build_context_proposals, configure_workflow_logging, workflow_paths, logged_step,
    write_approval_workbook,
)
from dq_agent.query_engine import QueryGuard, SQLCompiler, allowed_tables_for_rule
from dq_agent.relationships import (
    apply_relationship_precedence, calculate_relationship_confidence,
    candidates_to_context_records, discover_relationship_candidates, feature_flag_for_pair,
    load_relationship_configuration, load_sample_metadata, metadata_index,
    parent_profile_sql, profile_key, relationship_rule, trusted_relationship_candidates,
    validate_relationship_metadata,
)
from dq_agent.reporting import write_json

In [ ]:
RUN_ID = 'relationship_discovery_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
USE_SAMPLE_METADATA = True  # False reads enabled targets and dimensions from BigQuery
PROFILE_LIVE_VALUES = False  # Set True only after reviewing candidate SQL/cost limits
config = load_app_config(ROOT)
paths = workflow_paths(config, RUN_ID, 'relationships')
logger = configure_workflow_logging(paths['log'], config.project.log_level)
relationship_config = load_relationship_configuration(config)
business_context = load_business_context(config)
guard = QueryGuard()

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_RELATIONSHIP_CONFIGURATION'):
    if USE_SAMPLE_METADATA:
        sample = load_sample_metadata(ROOT / 'examples/relationship_sample_metadata.yaml')
        metadata = metadata_index(sample)
        pairs = [
            TablePair(pair_id='sample_fact_sales', mode='bigquery_only', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_sales'),
            TablePair(pair_id='sample_dim_market', mode='bigquery_only', target_project='your-gcp-project', target_dataset='analytics', target_table='dim_market'),
        ]
        parent_profiles = sample.get('parent_profiles', {})
        relationship_profiles = sample.get('relationship_profiles', {})
        connector = None
    else:
        pairs = load_table_pairs(config)
        if not pairs:
            raise ValueError('Enable at least one table in table_mappings.xlsx')
        connector = BigQueryConnector(config.project.query_limits, pairs[0].target_project)
        table_names = {pair.target_name for pair in pairs}
        table_names.update(dimension.table for dimension in relationship_config.dimensions.values() if dimension.enabled)
        table_names.update(item.parent_table for item in relationship_config.custom_relationships if item.enabled)
        metadata = {}
        for table_name in sorted(table_names):
            item = connector.get_table_metadata(table_name)
            metadata[table_name] = item
        parent_profiles, relationship_profiles = {}, {}
    logger.info('LOAD_RELATIONSHIP_CONFIGURATION pairs=%s metadata_tables=%s sample=%s', len(pairs), len(metadata), USE_SAMPLE_METADATA)
display(pd.DataFrame([{'pair_id': pair.pair_id, 'target_table': pair.target_name} for pair in pairs]))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'RETRIEVE_APPROVED_RELATIONSHIPS'):
    trusted = read_context(config, logger=logger)
    logger.info('RETRIEVE_APPROVED_RELATIONSHIPS records=%s', 0 if trusted.empty else len(trusted[trusted.context_type == 'relationship']))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'APPLY_FEATURE_FLAGS'):
    pair_by_id = {pair.pair_id: pair for pair in pairs}
    feature_flags = []
    for pair in pairs:
        table_context = business_context.get('tables', {}).get(pair.pair_id, {})
        enabled, source = feature_flag_for_pair(pair, relationship_config, table_context)
        feature_flags.append({'pair_id': pair.pair_id, 'enabled': enabled, 'source': source})
        logger.info('APPLY_FEATURE_FLAGS pair_id=%s enabled=%s source=%s', pair.pair_id, enabled, source)
display(pd.DataFrame(feature_flags))

with logged_step(logger, paths['checkpoint'], 'DISCOVER_RELATIONSHIP_CANDIDATES'):
    discovered = []
    for pair in pairs:
        child_metadata = metadata.get(pair.target_name) or metadata.get(pair.pair_id)
        if not child_metadata:
            raise ValueError(f'Metadata missing for {pair.target_name}')
        table_context = business_context.get('tables', {}).get(pair.pair_id, {})
        candidates, flag = discover_relationship_candidates(pair, child_metadata, relationship_config, table_context)
        candidates.extend(trusted_relationship_candidates(pair, trusted))
        candidates = apply_relationship_precedence(candidates)
        discovered.extend(candidates)
        logger.info('DISCOVER_RELATIONSHIP_CANDIDATES pair_id=%s enabled=%s source=%s candidates=%s', pair.pair_id, flag['enabled'], flag['source'], len(candidates))
display(pd.DataFrame(discovered)[['relationship_id','pair_id','child_columns','parent_table','parent_columns','origin','match_type','ambiguous']])

In [ ]:
with logged_step(logger, paths['checkpoint'], 'VALIDATE_RELATIONSHIP_METADATA'):
    validated = []
    for candidate in discovered:
        checked = validate_relationship_metadata(candidate, metadata)
        validated.append(checked)
        logger.info('VALIDATE_RELATIONSHIP_METADATA candidate=%s status=%s errors=%s', checked['candidate_id'], checked['metadata_status'], checked['metadata_errors'])
display(pd.DataFrame(validated)[['relationship_id','child_columns','parent_columns','metadata_status','metadata_errors','type_pairs']])

In [ ]:
with logged_step(logger, paths['checkpoint'], 'PROFILE_RELATIONSHIP_VALUES'):
    profiled = []
    profile_cache = {}
    for candidate in validated:
        pair = pair_by_id[candidate['pair_id']]
        parent_sql = parent_profile_sql(candidate)
        rule = relationship_rule(candidate)
        table_context = business_context.get('tables', {}).get(pair.pair_id, {})
        base_filters = list(table_context.get('filters', {}).get('target', []))
        relationship_sql = SQLCompiler('bigquery').compile_rule(rule, pair, 'target', base_filters)
        candidate['parent_profile_sql'] = parent_sql
        candidate['relationship_sql'] = relationship_sql
        if USE_SAMPLE_METADATA:
            parent_profile = parent_profiles.get(profile_key(candidate, parent=True), {})
            relationship_profile = relationship_profiles.get(profile_key(candidate), {})
        elif PROFILE_LIVE_VALUES and relationship_config.settings.profile_values and candidate['metadata_status'] == 'PASS':
            parent_cache_key = profile_key(candidate, parent=True)
            if parent_cache_key not in profile_cache:
                checked_parent_sql = guard.validate(parent_sql, 'bigquery', {candidate['parent_table']})
                profile_cache[parent_cache_key] = connector.execute(checked_parent_sql).frame.iloc[0].to_dict()
            parent_profile = profile_cache[parent_cache_key]
            checked_relationship_sql = guard.validate(relationship_sql, 'bigquery', allowed_tables_for_rule(rule, pair, 'target'))
            relationship_profile = connector.execute(checked_relationship_sql).frame.iloc[0].to_dict()
        else:
            parent_profile, relationship_profile = {}, {}
        candidate['loaded_parent_profile'] = parent_profile
        candidate['loaded_relationship_profile'] = relationship_profile
        profiled.append(candidate)
        logger.info('PROFILE_RELATIONSHIP_VALUES candidate=%s parent=%s relationship=%s', candidate['candidate_id'], parent_profile, relationship_profile)
    if connector:
        connector.close()

In [ ]:
with logged_step(logger, paths['checkpoint'], 'CALCULATE_RELATIONSHIP_CONFIDENCE'):
    scored = []
    for candidate in profiled:
        candidate = calculate_relationship_confidence(
            candidate, candidate['loaded_parent_profile'], candidate['loaded_relationship_profile'],
            config.project.confidence.auto_accept, config.project.confidence.review,
            relationship_config.settings.minimum_match_rate,
        )
        scored.append(candidate)
        logger.info('CALCULATE_RELATIONSHIP_CONFIDENCE candidate=%s score=%s decision=%s evidence=%s', candidate['candidate_id'], candidate['confidence'], candidate['decision'], candidate['confidence_evidence'])
    candidates = pd.DataFrame(scored)
display(candidates[['relationship_id','child_columns','parent_table','parent_columns','origin','confidence','decision','ambiguous','match_rate','confidence_evidence']])

In [ ]:
with logged_step(logger, paths['checkpoint'], 'CREATE_RELATIONSHIP_APPROVAL'):
    review_records = candidates_to_context_records(scored, config, RUN_ID)
    proposals = build_context_proposals(review_records, trusted, RUN_ID)
    approval_file = None
    if not proposals.empty:
        approval_file = paths['pending'] / f'relationship_approvals_{RUN_ID}.xlsx'
        write_approval_workbook(approval_file, proposals)
        logger.info('CREATE_RELATIONSHIP_APPROVAL output=%s records=%s', approval_file, len(proposals))
    write_json(paths['logs'] / 'relationship_candidates.json', scored)
    candidates.to_csv(paths['logs'] / 'relationship_candidates.csv', index=False)
    print('Discovery run:', RUN_ID, 'Approval file:', approval_file)

Use `02_approval_processing.ipynb` for relationship approvals. Then rerun this notebook so approved relationships are retrieved with origin `trusted` before execution.